## **What data structure to use for iterating over the variants?**

### **First implementation** (Single Chromosome)
The first implementation was to use a queue that stored both the variant info, as well as the features to be used in calculating the statistics. 

The problem with this approach was that if we were to process multiple BAM files, we would need create the list of variants over again as we are popping variants once they are processed from the front of the queue. The approach for this would be to separate the variant info into its own vector; then for each variant, create a struct that has a lifetime non mutable reference to the variant info, and another field with the features. Lifetime notation is needed as the compiler needs to know beforehand which fields. [Need to further understand lifetimes.]

However, since multiple BAM processing is not necessary, we can instead keep it as it is currently.

In the future, it may be better to have a function that initializes the Variants vector in main. Then have another function that would iterate over however many read files are supplied, and for each of them create the struct with the features field and lifetime notation for the VariantsInfo.
Is reinitalizing the Variants vector that big of a deal though? Testing shows that for `HG00096.mapped.ILLUMINA.bwa.GBR.exome.20120522.lossless.bam.cram` variant_parsing only took ~1 second.


```mermaid
flowchart TD
    subgraph Implementation if multi-bam processing is needed

    var_vector_1@{ shape: bow-rect, label: "Vector of variant_info
    [Vector]
    chrom: String,
	pos: u64,
	refr: String,
	alt: String,
    vartype: VarType," }

    var_vector_1 --> var_queue_2@{ shape: bow-rect, label: "Variants Vector
    [Vector Queue (vecdeque)]
    variant: &variant_info
    features: LocusFeatures," }

    end

    subgraph Current implementation

    var_queue_1@{ shape: bow-rect, label: "Variants Queue
    [Vector Queue (vecdeque)]
    chrom: String,
	pos: u64,
	refr: String,
	alt: String,
    vartype: VarType,
    features: LocusFeatures," }

    end 
```

### **Second implementation attempt** (Multi-chromosome)
The first approach only works if all the variants are from a single chromosome. Since we are no longer using the variant pileup approach, and instead iterating over reads and seeing if they overlap with the variants, we need to instead call `fetch` in the bam reader. `fetch` is only able to fetch reads within a given position range for a single chromosome, therefore we needed to somehow separate the variants so that we can call `fetch` a subset.

With the `VecDeque` already, one thing we could do would be to calculate the number of variants for each chromosome and save this in a vector; then we could subset the queue by [0..number of variants], processing the chromosome subset which will be popped once completed. Then we repeat this for the rest of the chromosomes, since the start of the queue will be the start of a new chromosome with the range ending in the next number of variants. However, since VecDeque is implemented as a ring buffer, the elements inside may not be contiguous if they wrap wround the end of the physical buffer and so we cannot just simply slice a subset (Can do if we use `as_slice` or `make_contiguous` but unsure about performance/feasability). 

We could use either `drain` or `split_off`: 
- `drain` will remove a given range from the queue and return them as an iterator; however, since we do not want to consume the variant after iterating over it as it could align to another read, we would have to add the trait implementation `peekable` which has methods that can look at the next element in the iterator without consuming it. Downside of this approach is that the remaining elements of the queue will need to be shifted to the front of the queue [Time complexity: O(M) where M is the remaining variants in the queue | Space complexity: O(1) as we do not need to any new allocations].
- `split_off` will split the given range into a new `VecDeque`. The problem with this is that there is O(N) time complexity, where N is the all the elements in the queue, as we need to allocate the split to a new chunk of memory, while also shifting the remaining elements. There is also O(S) space complexity, where S is the number of split elements.

Another approach would be to separate the single `VecDeque` into separate queues for each chromosome when initializing it. For example, if the variants list contains chromosomes 1, 2, 3 and 4, then we would create a structure for each of these containing the chromosome and a `VecDeque`. This would essentially create a bucket of variants for each chromosome, which we could then append to a vector; as we are assuming the variants are sorted by at least chromosome and position, we do not have to do any sorting. I have implemented this approach as it is the easiest to understand without the complexity of splitting the queue if we want to implement multithreading later.

```mermaid
flowchart TD
    subgraph Current implementation

    var_queue_1@{ shape: bow-rect, label: "Variants Queue
    [VecDeque]
    chrom: String,
	pos: u64,
	refr: String,
	alt: String,
    vartype: VarType,
    features: LocusFeatures," }

    var_queue_1 --> bucket@{ shape: docs, label: "Chromosome buckets 
    [Struct]
    chrom: String,
    variants: Variants Queue" }

    bucket --> parse@{ shape: bow-rect, label: "Parsed variants
    [Vec]
    chroms: Chromosome bucket" }

    end 
```


### **Future implementation** (Multi-threading)

If we were to attempt to implement mulit-threading with the previous data structure in place, the best approach just be to chunk the chromosomes and use and process them with the number of threads given when available. 

    // ADD LOOP FOR CHROM HERE 
    // Since chrom/variants in sorted order
    // start of chrom in queue should be 0 as first entry should be first instance of variant for a chromosome
    // end of chrom in queue should be the variant count
    // therefore we should only iterate over [0 .. variant_count]

Big question is how are we to split writing of the files?
- Writing the csv:
    - Create temp files for all threads, with ordered suffixes somehow? Then merge at end? Prediction is I/O cost will not be worth it depending on how many threads spawned?
    - Store stats in memory and write all at once after finishing. Need to calculate space complexity of variant statistics storage
- IF we do not need to pop from queue, it would be easier to iterate over a vector of chromosomes from all variants. We could also iterate over a `VecDeque`, but it may be complicated with the parallelization, unless each thread splits off its own queues. 

NOTE: as discussing with Bernie, he made a point I overlooked, in that calling fetch over the variants range may fetch in a lot of unused reads if variants are spaced apart; discussion led to deciding to create bins of variants if they are greater than some average read length time some value. (i.e. if variants are farther apart than n x avg.read.length then we bin to be fetched and processed separately.)

Also suggested just initializing the list of variants first which I agree with; then we can make a separate module/function that bins the variants so that it is easier to refactor in the future in case. The extra processing time of binning over an already initialized vector compared to binning at the same time is probably not that much. We should then implement the separation of variant info and variant stats as suggested in the first implementation attempt.

For binning, we can do something similar with our current implementation. Iterate over the variants and bin them together until they are spaced too far apart, which we then put into a new bin. Structure will still be chrombuckets in vector, but seems like there will be a lot more chrombuckets. (We should test how many before/after).

For multithreading we can have a new heuristic; we can implement a simple greedyish algorithm at first:
- Get the total number of variants and the number of threads wanted
- Let n = variants divided by number of threads
- Iterate over lengths of each bin; when bin is ~same size as n, set them aside for a thread to handle
- Continue until all threads except last are filled (last bin will just have leftover)

### **Binning implementation**

Trying the implementation where we create new bins if variants are > (2x average read length) apart, we create hundreds of thousands of bins; calling fetch on all these bins is extremely expensive due to the constant I/O of the reads file (I/O is the most expensive part of this program as Ben suggested), as when testing we found that it was taking longer than 1hr and stopped the process. Therefore, maybe it would be better to make the distance between bins larger. We should do some testing of some hyperparameters for the gap between variants. Breakpoints such as 50kB, 100kB, 500kB, 1Mb should be tested. Using the full HG0096 exome data we found that without binning within chromosomes, there were 71 bins corresponding to the all the unique chromosome labels. Setting the gap to 100kB created 509, while setting it to 1Mb created 95 bins. Both of these were timed simlarly, but we need to do tests that had many variants at the start and end of chromosomes to get a better idea of how much more it would save. 

Testing using only the first and last of each chromosome (this filtered vcf thus contains 131 variants compared to the full 933,141 variants), we find that for no gap the program takes around ~1m40s, while the python varlap runs in ~3s. Implementing a 100kb gap for rust reduces the time to ~2s.